
## 1. Setup and Dependencies

This section prepares the Python environment by installing and importing the libraries required for PDF text extraction, data processing, and validation.

**Libraries:** PyMuPDF, Pandas, and Regular Expressions.

**Expected output:** Successful environment initialization.



In [23]:

%pip install -q pymupdf pandas

import re
from pathlib import Path

import fitz
import pandas as pd
from IPython.display import display

print("Origin | Data Preparation")
print("PyMuPDF version:", fitz.VersionBind)
print("Setup completed successfully!")


Origin | Data Preparation
PyMuPDF version: 1.28.2
Setup completed successfully!



## 2. Upload and Load PDF

Upload the original graduation projects PDF from Princess Nourah bint Abdulrahman University.

The document contains graduation project titles, abstracts, and related information from the 2024–2025 academic year.

This step verifies that the PDF is successfully loaded and reports its total number of pages.

**Expected output:** 114 PDF pages.



In [24]:

from google.colab import files

print("Upload the graduation projects PDF")

uploaded = files.upload()

assert len(uploaded) == 1, "Please upload one PDF file."

filename = next(iter(uploaded))

assert filename.lower().endswith(".pdf"), (
    "The uploaded file must be a PDF."
)

pdf_path = Path("/content") / filename
doc = fitz.open(str(pdf_path))

print("\nPDF INFORMATION")
print("=" * 45)
print("File:", filename)
print("Total pages:", len(doc))
print("PDF loaded successfully!")


Upload the graduation projects PDF


Saving a2865b46-f8dd-43c7-89bb-61cc6ae52ca7.pdf to a2865b46-f8dd-43c7-89bb-61cc6ae52ca7 (2).pdf

PDF INFORMATION
File: a2865b46-f8dd-43c7-89bb-61cc6ae52ca7 (2).pdf
Total pages: 114
PDF loaded successfully!



## 3. PDF Inspection and Project Page Detection

Inspect the extracted text and identify pages containing graduation project abstracts.

Pages are classified using the standalone "Abstract" heading to distinguish actual project records from introductory pages, department covers, and other non-project content.

This ensures that only valid project pages are included in the dataset.

**Expected output:** 102 project pages.


In [25]:

# Inspect representative PDF pages

sample_pages = [1, 3, 5, 10, 26]

print("PDF TEXT INSPECTION")
print("=" * 55)

for page_num in sample_pages:
    text = doc[page_num - 1].get_text(
        "text", sort=True
    )

    print(f"\n--- PAGE {page_num} ---")
    print(text[:350])
    print("-" * 55)


# Identify actual project pages

abstract_pattern = re.compile(
    r"(?im)^[ \t]*Abstract[ \t]*$"
)

project_pages = []

for page_num, page in enumerate(doc, start=1):
    text = page.get_text("text", sort=True)

    if abstract_pattern.search(text):
        project_pages.append(page_num)


non_project_pages = [
    page_num
    for page_num in range(1, len(doc) + 1)
    if page_num not in project_pages
]

print("\nPAGE DETECTION SUMMARY")
print("=" * 55)

print("Total PDF pages:", len(doc))
print("Detected projects:", len(project_pages))
print("Non-project pages:", non_project_pages)


PDF TEXT INSPECTION

--- PAGE 1 ---
Princess Nourah Bint Abdulrahman University
College of Computer and Information Sciences





            Graduation Projects
           Book of Abstracts
               2024-2025
-------------------------------------------------------

--- PAGE 3 ---
Contents


Department of Computer Science - Bachelor of Science in Computer Science
 .......................................................................................................................... 4

Department of Computer Science - Bachelor of Science in Artificial
Intelligence ...........................................................
-------------------------------------------------------

--- PAGE 5 ---
              Automatic Plant Watering and Tracking System

  Refan Hussein BinMusaad, Rand Aiman Alsaigal, Hamdah Hamad Alanazi,
                       Lana Khalid Alrumaih

                       Supervisor: Dr. Ahmed Ibrahim

Abstract

Many people struggle with watering their plants re


## 4. Initial Project Extraction

Extract the project header and abstract from each identified project page.

The header contains the project title, student names, and supervisor information. The abstract contains the project description and, where available, keywords.

The extracted information is stored in a preliminary DataFrame while preserving the original PDF page numbers.

**Output:** `raw_df`


In [26]:

records = []

for page_num in project_pages:

    text = doc[page_num - 1].get_text(
        "text", sort=True
    )

    match = abstract_pattern.search(text)

    if match is None:
        continue

    header = text[:match.start()].strip()
    abstract = text[match.end():].strip()

    records.append({
        "source_page": page_num,
        "header_text": header,
        "abstract_text": abstract
    })


raw_df = pd.DataFrame(records)

raw_df = raw_df.sort_values(
    "source_page"
).reset_index(drop=True)

print("INITIAL EXTRACTION")
print("=" * 50)

print("Extracted records:", len(raw_df))
print(
    "Empty abstracts:",
    raw_df["abstract_text"].eq("").sum()
)

print("\nSAMPLE PROJECT HEADERS")

for _, row in raw_df.head(3).iterrows():
    print("\nPage:", row["source_page"])
    print(row["header_text"][:350])
    print("-" * 50)


INITIAL EXTRACTION
Extracted records: 102
Empty abstracts: 0

SAMPLE PROJECT HEADERS

Page: 5
Automatic Plant Watering and Tracking System

  Refan Hussein BinMusaad, Rand Aiman Alsaigal, Hamdah Hamad Alanazi,
                       Lana Khalid Alrumaih

                       Supervisor: Dr. Ahmed Ibrahim
--------------------------------------------------

Page: 6
CourseAssist: CLO Performance Assistant “E-CPA”

   Raghad Khalid Aldreweesh, Renad Abdulrahman Ajina, Shahad Ibrahim

                         Alharbi, Yara melaifi Alanazi

                       Supervisor: Dr. Ahmed Ibrahim
--------------------------------------------------

Page: 7
Mobile Application for Deaf and Mute individuals. (WASIF)

  Reemas turki alaqil, Abeer Sultan Alabdullatif, Haifa Abdulkarim Alkhalaf,

                       Layan Yahya Almegbil.

                  Supervisor: Dr. Eman Abdullah Aldakheel
--------------------------------------------------



## 5. Data Cleaning and Structured Dataset Creation

Transform the extracted information into a structured dataset.

The preprocessing includes:

- Extracting project titles from their headers.
- Removing repeated PDF footers and unnecessary whitespace.
- Separating keywords from project abstracts.
- Assigning the appropriate academic department.
- Generating unique project identifiers.
- Preserving the academic year and original source page.

A temporary review flag is included to identify potentially irregular title formatting.

**Output:** `projects_df`



In [27]:

# Department page ranges

department_ranges = [
    (5, 37, "Computer Science"),
    (39, 49, "Artificial Intelligence"),
    (51, 73, "Information Systems"),
    (75, 84, "Data Science and Analysis"),
    (86, 102, "Information Technology"),
    (104, 111, "Cybersecurity")
]


def get_department(page):
    for start, end, department in department_ranges:
        if start <= page <= end:
            return department
    return "Unknown"


def clean_text(text):
    return re.sub(r"\s+", " ", text).strip()


def extract_title(header):

    before_supervisor = re.split(
        r"(?im)^[ \t]*Supervisor[ \t]*:",
        header,
        maxsplit=1
    )[0].strip()

    paragraphs = re.split(
        r"\n[ \t]*\n",
        before_supervisor
    )

    title = clean_text(paragraphs[0])

    needs_review = (
        len(paragraphs) < 2 or not title
    )

    return title, needs_review


def remove_footer(text):

    pattern = (
        r"(?im)^[ \t]*Graduation Projects"
        r"[ \t]+2024[ \t]*[–—-][ \t]*2025"
        r"[ \t]*\d+[ \t]*$"
    )

    return re.sub(pattern, "", text).strip()


def split_abstract_keywords(text):

    text = remove_footer(text)

    keyword_pattern = (
        r"(?im)^[ \t]*Key[ -]?words?\b"
        r"[ \t]*[:：]?[ \t]*"
    )

    match = re.search(keyword_pattern, text)

    if match:
        abstract = text[:match.start()]
        keywords = text[match.end():]
    else:
        abstract = text
        keywords = ""

    return (
        clean_text(abstract),
        clean_text(keywords)
    )


# Build the structured dataset

projects = []

for idx, row in raw_df.iterrows():

    title, needs_review = extract_title(
        row["header_text"]
    )

    abstract, keywords = split_abstract_keywords(
        row["abstract_text"]
    )

    projects.append({
        "project_id": f"P{idx + 1:03d}",
        "title": title,
        "abstract": abstract,
        "keywords": keywords,
        "department": get_department(
            row["source_page"]
        ),
        "year": "2024-2025",
        "source_page": row["source_page"],
        "needs_review": needs_review
    })


projects_df = pd.DataFrame(projects)

print("STRUCTURED DATASET CREATED")
print("=" * 50)

print("Total projects:", len(projects_df))
print("Columns:", projects_df.columns.tolist())

display(projects_df.head())


STRUCTURED DATASET CREATED
Total projects: 102
Columns: ['project_id', 'title', 'abstract', 'keywords', 'department', 'year', 'source_page', 'needs_review']


,project_id,title,abstract,keywords,department,year,source_page,needs_review
0,P001,Automatic Plant Watering and Tracking System,"Many people struggle with watering their plants regularly, which usually leads to plants becoming unhealthy or dying. To address this common issue, it is important to monitor basic factors such as soil moisture and temperature in order to determine the proper time and amount of water needed. Since manual watering is not always consistent or reliable, this automatic plant watering system is designed to assist gardeners by providing an accurate irrigation based on real time weather conditions and the specific water requirements of each plant. The system includes a monitoring mechanism that helps ensure proper and efficient water usage, reducing water waste and supporting healthy plant growth. Additionally, the application includes a built-in encyclopedia that offers detailed information about a variety of plant types, helping users to understand their plant’s water needs, caring details and other useful facts. This solution makes it easier to take care for plants, especially when they are located away from owners.","Moisturem Air humidity, AWTS system, Waterin, Plants .",Computer Science,2024-2025,5,False
1,P002,CourseAssist: CLO Performance Assistant “E-CPA”,"The rapid evolution of educational technology necessitates innovative approaches to enhance the assessment of student performance and course effectiveness. Our project addresses a significant issue present when faculty members spend an excessive amount of time completing the necessary documentation to achieve course objectives. To tackle this problem, we developed ""Alama,"" an automated mobile application that streamlines the traditional Excel data entry process. This application calculates Course Learning Outcomes (CLOs) and Program Learning Outcomes (PLOs) automatically, generating performance reports in moments rather than relying on manual entry. Alama not only mitigates the inefficiencies of conventional assessment methods, which can lead to inaccuracies and impede effective decision-making, but also integrates both quantitative and qualitative data. This holistic approach enhances the evaluation of educational processes, ultimately improving the learning experience for students. In conclusion, the implementation of Alama presents a significant advancement in academic assessment, offering both advantages—such as time savings and enhanced accuracy—and potential drawbacks, including the initial learning curve for faculty. Overall, Alama holds the promise of transforming educational assessment practices, leading to better educational outcomes.","Program Learning Outcomes, Course Learning Outcomes, Performance Assistant, Formative, Summative",Computer Science,2024-2025,6,False
2,P003,Mobile Application for Deaf and Mute individuals. (WASIF),"Our System, Wasif, offers a communication application for deaf and mute individuals, we aim to develop cross-platform application that bridges these communication gaps by translating spoken language into sign language symbols and written text, while converting sign language gestures into audible speech. Wasif application analyzes hand and facial movements in real-time, ensuring accurate and efficient communication. Built using robust frameworks which is TensorFlow, and OpenCV, seeks to empower deaf and mute individuals to fully engage in all aspects of life, it is have been validated through rigorous testing with target user groups, ensuring that it meets the real-world needs.","Arabic Sign Language, Sign Language Gestures into Audial Speech, Real-time Facial and Hand Movement Tracking, Deaf and mute.",Computer Science,2024-2025,7,False
3,P004,Pet Care Application System (Anees),"Pets have always played a vital role in human companionship, offering emotional support, reducing stress, and enhancing overall well-being. As pet ownership continues to grow in Saudi Arabia, so there is a high demand for accessib


## 6. Data Quality Validation

Evaluate the quality and consistency of the extracted dataset before exporting it.

Validation includes checking missing values, duplicate titles, unknown departments, keyword coverage, abstract lengths, and project distribution across departments.

Project titles at department boundaries are inspected to verify correct classification.

Potential extraction issues and missing keywords are also reviewed manually.

**Expected output:** 102 valid project records distributed across six departments.


In [28]:

print("DATA QUALITY VALIDATION")
print("=" * 55)

print("Total projects:", len(projects_df))

print(
    "Missing titles:",
    projects_df["title"].eq("").sum()
)

print(
    "Missing abstracts:",
    projects_df["abstract"].eq("").sum()
)

print(
    "Duplicate titles:",
    projects_df["title"].duplicated().sum()
)

print(
    "Unknown departments:",
    projects_df["department"].eq("Unknown").sum()
)

print(
    "Titles requiring review:",
    projects_df["needs_review"].sum()
)

print(
    "Projects with keywords:",
    projects_df["keywords"].ne("").sum()
)

print(
    "Projects without keywords:",
    projects_df["keywords"].eq("").sum()
)


# Department distribution

print("\nPROJECTS BY DEPARTMENT")
print("=" * 55)

print(projects_df["department"].value_counts())


# Abstract length statistics

word_counts = (
    projects_df["abstract"].str.split().str.len()
)

print("\nABSTRACT LENGTH STATISTICS")
print("=" * 55)

print("Minimum words:", word_counts.min())
print("Maximum words:", word_counts.max())
print("Average words:", round(word_counts.mean(), 1))


# Check department boundaries

boundary_pages = [
    5, 37, 39, 49, 51, 73,
    75, 84, 86, 102, 104, 111
]

print("\nDEPARTMENT BOUNDARY CHECK")

display(
    projects_df.loc[
        projects_df["source_page"].isin(
            boundary_pages
        ),
        [
            "project_id",
            "title",
            "department",
            "source_page"
        ]
    ]
)


# Review missing keywords

print("\nPROJECTS WITHOUT KEYWORDS")

display(
    projects_df.loc[
        projects_df["keywords"].eq(""),
        ["project_id", "title", "source_page"]
    ]
)


# Inspect remaining keyword mentions

print("\nKEYWORD MENTIONS WITHIN ABSTRACTS")

mask = projects_df["abstract"].str.contains(
    r"\bkeywords?\b",
    case=False,
    regex=True,
    na=False
)

for _, row in projects_df[mask].iterrows():

    print("\nProject:", row["title"])
    print("Page:", row["source_page"])

    for match in re.finditer(
        r"\bkeywords?\b",
        row["abstract"],
        re.I
    ):
        start = max(0, match.start() - 100)
        end = min(
            len(row["abstract"]),
            match.end() + 100
        )

        print(row["abstract"][start:end])


# Check for short abstracts or leftover footers

suspicious = projects_df[
    (word_counts < 40)
    | projects_df["abstract"].str.contains(
        r"Graduation Projects\s*2024|Supervisor\s*:",
        case=False,
        regex=True,
        na=False
    )
]

print("\nPOTENTIALLY SUSPICIOUS RECORDS:", len(suspicious))

display(
    suspicious[
        ["project_id", "title", "source_page"]
    ]
)


# Final automated assertions

assert len(projects_df) == 102
assert projects_df["project_id"].is_unique
assert projects_df["title"].ne("").all()
assert projects_df["abstract"].ne("").all()
assert projects_df["department"].ne("Unknown").all()

print("\nAutomated validation completed successfully!")
print("Manual review of displayed samples is still required.")


DATA QUALITY VALIDATION
Total projects: 102
Missing titles: 0
Missing abstracts: 0
Duplicate titles: 0
Unknown departments: 0
Titles requiring review: 0
Projects with keywords: 101
Projects without keywords: 1

PROJECTS BY DEPARTMENT
department
Computer Science             33
Information Systems          23
Information Technology       17
Artificial Intelligence      11
Data Science and Analysis    10
Cybersecurity                 8
Name: count, dtype: int64

ABSTRACT LENGTH STATISTICS
Minimum words: 61
Maximum words: 277
Average words: 171.8

DEPARTMENT BOUNDARY CHECK


,project_id,title,department,source_page
0,P001,Automatic Plant Watering and Tracking System,Computer Science,5
32,P033,RASID | Road defect detection and classification,Computer Science,37
33,P034,Visualize,Artificial Intelligence,39
43,P044,Nabeeh : Artificial Intelligence-Based System for Attention monitoring of students,Artificial Intelligence,49
44,P045,SaudiWay,Information Systems,51
66,P067,MALI – Expense Tracker Application,Information Systems,73
67,P068,Bridging Communication Gaps: A system for Arabic sign language translation,Data Science and Analysis,75
76,P077,AI-Driven Serious Games for ADHD Treatment,Data Science and Analysis,84
77,P078,A Software-Defined Networking Approach to Link Failure Solutions in Cloud Environments,Information Technology,86
93,P094,Image Forgery Detection Website: ForgeFinder,Information Technology,102



PROJECTS WITHOUT KEYWORDS


,project_id,title,source_page
44,P045,SaudiWay,51



KEYWORD MENTIONS WITHIN ABSTRACTS

Project: AI-Powered Platform for Monitoring and Analyzing Consumer Sentiments in the Cosmetics Industry
Page: 77
nsights. The platform includes an integrated search engine that allows users to find products using keywords or hashtags, connecting directly to a text-based social media platform, such as Twitter. The system

POTENTIALLY SUSPICIOUS RECORDS: 0


,project_id,title,source_page



Automated validation completed successfully!
Manual review of displayed samples is still required.



## 7. Final Dataset Export

Export the validated and cleaned graduation projects dataset as a CSV file.

The temporary review column is removed, and the final dataset contains the following attributes:

- Project ID
- Project Title
- Abstract
- Keywords
- Department
- Academic Year
- Source Page

The resulting dataset will be used for exploratory data analysis, information retrieval experiments, and semantic similarity evaluation in the Origin project.

**Output:** `projects_clean.csv`


In [29]:

# Prepare final dataset

final_df = projects_df.drop(
    columns=["needs_review"]
).copy()


# Validate final data

assert len(final_df) == 102
assert final_df["project_id"].is_unique
assert final_df["title"].ne("").all()
assert final_df["abstract"].ne("").all()


# Export to CSV

output_path = Path("/content/projects_clean.csv")

final_df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)


print("FINAL DATASET EXPORTED")
print("=" * 50)

print("File:", output_path.name)
print("Total projects:", len(final_df))
print("Total columns:", len(final_df.columns))

print("\nDataset columns:")
print(final_df.columns.tolist())

display(final_df.head())


# Download CSV

files.download(str(output_path))


FINAL DATASET EXPORTED
File: projects_clean.csv
Total projects: 102
Total columns: 7

Dataset columns:
['project_id', 'title', 'abstract', 'keywords', 'department', 'year', 'source_page']


,project_id,title,abstract,keywords,department,year,source_page
0,P001,Automatic Plant Watering and Tracking System,"Many people struggle with watering their plants regularly, which usually leads to plants becoming unhealthy or dying. To address this common issue, it is important to monitor basic factors such as soil moisture and temperature in order to determine the proper time and amount of water needed. Since manual watering is not always consistent or reliable, this automatic plant watering system is designed to assist gardeners by providing an accurate irrigation based on real time weather conditions and the specific water requirements of each plant. The system includes a monitoring mechanism that helps ensure proper and efficient water usage, reducing water waste and supporting healthy plant growth. Additionally, the application includes a built-in encyclopedia that offers detailed information about a variety of plant types, helping users to understand their plant’s water needs, caring details and other useful facts. This solution makes it easier to take care for plants, especially when they are located away from owners.","Moisturem Air humidity, AWTS system, Waterin, Plants .",Computer Science,2024-2025,5
1,P002,CourseAssist: CLO Performance Assistant “E-CPA”,"The rapid evolution of educational technology necessitates innovative approaches to enhance the assessment of student performance and course effectiveness. Our project addresses a significant issue present when faculty members spend an excessive amount of time completing the necessary documentation to achieve course objectives. To tackle this problem, we developed ""Alama,"" an automated mobile application that streamlines the traditional Excel data entry process. This application calculates Course Learning Outcomes (CLOs) and Program Learning Outcomes (PLOs) automatically, generating performance reports in moments rather than relying on manual entry. Alama not only mitigates the inefficiencies of conventional assessment methods, which can lead to inaccuracies and impede effective decision-making, but also integrates both quantitative and qualitative data. This holistic approach enhances the evaluation of educational processes, ultimately improving the learning experience for students. In conclusion, the implementation of Alama presents a significant advancement in academic assessment, offering both advantages—such as time savings and enhanced accuracy—and potential drawbacks, including the initial learning curve for faculty. Overall, Alama holds the promise of transforming educational assessment practices, leading to better educational outcomes.","Program Learning Outcomes, Course Learning Outcomes, Performance Assistant, Formative, Summative",Computer Science,2024-2025,6
2,P003,Mobile Application for Deaf and Mute individuals. (WASIF),"Our System, Wasif, offers a communication application for deaf and mute individuals, we aim to develop cross-platform application that bridges these communication gaps by translating spoken language into sign language symbols and written text, while converting sign language gestures into audible speech. Wasif application analyzes hand and facial movements in real-time, ensuring accurate and efficient communication. Built using robust frameworks which is TensorFlow, and OpenCV, seeks to empower deaf and mute individuals to fully engage in all aspects of life, it is have been validated through rigorous testing with target user groups, ensuring that it meets the real-world needs.","Arabic Sign Language, Sign Language Gestures into Audial Speech, Real-time Facial and Hand Movement Tracking, Deaf and mute.",Computer Science,2024-2025,7
3,P004,Pet Care Application System (Anees),"Pets have always played a vital role in human companionship, offering emotional support, reducing stress, and enhancing overall well-being. As pet ownership continues to grow in Saudi Arabia, so there is a high demand for accessible, high-quality pet care servi

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>